In [ ]:
#| default_exp spec

In [ ]:
#| hide
from fastcore.test import *
from nbdev.showdoc import *
from tempfile import TemporaryDirectory
from fastcore.all import Path

Define one application specification for the platform freezer.

In [ ]:
#| export
from __future__ import annotations
import sys
from dataclasses import dataclass, field
from fastcore.all import Path, globtastic, groupby

In [ ]:
#| export
def tree(src, dest, skip=()):
    "Every file under `src` as freezer `(dest_dir, [files])` pairs, recursively."
    fs = globtastic(src, skip_folder_re=f'^({"|".join([*skip, "__pycache__"])})$', skip_file_re=r'^\.')
    dirs = groupby(fs, lambda f: str(Path(dest)/Path(f).parent.relative_to(src)))
    return sorted((d, sorted(v)) for d, v in dirs.items())

def mypyc_modules(site=None):
    "Return top-level mypyc extension modules from `site`."
    import sysconfig
    site = Path(site or sysconfig.get_paths()['purelib'])
    return (sorted(f.name.split('.')[0] for f in site.glob('*__mypyc*.so')) +
            sorted(f.name.split('.')[0] for f in site.glob('*__mypyc*.pyd')))

`tree` returns deterministic freezer data-file pairs and excludes hidden files, caches, and requested directories.

In [ ]:
#| hide
tmp = TemporaryDirectory(); src = Path(tmp.name)/'static'
for d in ('js', 'img', 'node_modules', 'empty'): (src/d).mkdir(parents=True)
(src/'app.css').write_text('a{}'); (src/'.DS_Store').write_text('')
(src/'js'/'app.js').write_text('//'); (src/'img'/'logo.png').write_text('x')
(src/'node_modules'/'left-pad.js').write_text('//')

2

In [ ]:
data = tree(src, 'demo/static', skip=['node_modules'])
[(dest, [Path(f).name for f in files]) for dest, files in data]

[('demo/static', ['app.css']),
 ('demo/static/img', ['logo.png']),
 ('demo/static/js', ['app.js'])]

In [ ]:
#| hide
test_eq([dest for dest, _ in data], ['demo/static', 'demo/static/img', 'demo/static/js'])
assert all(Path(f).is_file() for _, files in data for f in files), 'real paths, for the freezer to read'

`mypyc_modules` discovers version-specific compiled accelerators from the environment being frozen.

In [ ]:
site = Path(tmp.name)/'site'; site.mkdir()
for f in ('chardet__mypyc.cpython-313-darwin.so', 'numpy.cpython-313-darwin.so'): (site/f).write_text('')
mypyc_modules(site)

['chardet__mypyc']

In [ ]:
#| export
DEFAULT_EXCLUDES = ('tkinter', 'test', 'setuptools', 'wheel', 'py2app', 'py2exe',
                    'gi', 'PyQt5', 'PyQt6', 'PySide2', 'PySide6', 'clr', 'matplotlib')

GUI_MODULE = {'darwin': 'webview.platforms.cocoa', 'win32': 'webview.platforms.winforms',
              'linux': 'webview.platforms.gtk'}

`DEFAULT_EXCLUDES` removes build tools, tests, unused GUI backends, and large optional toolkits from frozen applications.

In [ ]:
#| export
@dataclass
class App:
    """A desktop app specification for platform-specific bundling."""
    name: str
    entry: str                                  # the script the bundle runs
    version: str = '0.0.0'
    icon: str = ''                              # an .icns/.ico; `kavacha.icons` makes one
    modern_icon: str = ''                       # a macOS 26 Icon Composer document
    identifier: str = ''                        # reverse-DNS bundle id; derived from `name` if unset
    summary: str = ''
    packages: list = field(default_factory=list)    # copied whole, not scanned
    includes: list = field(default_factory=list)    # single modules the scanner cannot reach
    excludes: list = field(default_factory=list)    # on top of `DEFAULT_EXCLUDES`
    grafted: list = field(default_factory=list)     # put in place after the freezer runs
    unreachable: list = field(default_factory=list) # archive prefixes nothing can read
    data: list = field(default_factory=list)        # `(dest_dir, [files])` pairs, as `tree` makes
    plist: dict = field(default_factory=dict)       # merged over the defaults below
    doc_types: list = field(default_factory=list)
    extras: list = field(default_factory=list)      # the extras a build environment installs
    min_macos: str = '11.0'
    category: str = 'public.app-category.developer-tools'

    def __post_init__(self):
        self.identifier = self.identifier or f'org.{self.name.lower()}.{self.name.lower()}'
    @property
    def bundle_name(self): return f'{self.name}.app' if sys.platform == 'darwin' else self.name
    def out(self, root): return Path(root)/'dist'/self.bundle_name
    def info_plist(self):
        "The Info.plist entries for this app."
        base = {
            'CFBundleName': self.name, 'CFBundleDisplayName': self.name,
            'CFBundleIdentifier': self.identifier,
            'CFBundleVersion': self.version, 'CFBundleShortVersionString': self.version,
            'CFBundleGetInfoString': self.summary or self.name,
            'LSApplicationCategoryType': self.category,
            'LSMinimumSystemVersion': self.min_macos,
            'NSHighResolutionCapable': True,
            'NSRequiresAquaSystemAppearance': False,
            'NSAppTransportSecurity': {'NSAllowsLocalNetworking': True},
            'LSEnvironment': {'LANG': 'en_US.UTF-8', 'PYTHONUTF8': '1'},
        }
        if self.doc_types: base['CFBundleDocumentTypes'] = self.doc_types
        return base | dict(self.plist)

    def excluded(self, platform=None):
        "The exclusions for this build and platform."
        others = [m for p, m in GUI_MODULE.items() if p != (platform or sys.platform)]
        return sorted({*DEFAULT_EXCLUDES, *others, *self.excludes})

    def py2app_options(self):
        "The py2app options for this spec."
        out = {'packages': list(self.packages), 'includes': list(self.includes),
               'excludes': self.excluded('darwin'), 'plist': self.info_plist(),
               'argv_emulation': False, 'semi_standalone': False, 'site_packages': True,
               'strip': True}
        if self.icon: out['iconfile'] = str(self.icon)
        return out

    def py2exe_options(self):
        "The py2exe options for this spec."
        return {'packages': list(self.packages), 'includes': list(self.includes),
                'excludes': self.excluded('win32'), 'bundle_files': 3, 'compressed': 1}

`App` contains freezer-independent application metadata and derives a lowercase identifier when none is supplied.

In [ ]:
app = App(name='Demo', entry='demo_app.py', version='1.2.3', summary='the demo app',
          icon='assets/Demo.icns', packages=['demo', 'fasthtml', 'uvicorn'],
          includes=['demo.cli'], data=data, extras=['desktop'])
app.identifier, app.bundle_name, str(app.out('/build/demo'))

('org.demo.demo', 'Demo', '/build/demo/dist/Demo')

In [ ]:
app.info_plist()

{'CFBundleName': 'Demo',
 'CFBundleDisplayName': 'Demo',
 'CFBundleIdentifier': 'org.demo.demo',
 'CFBundleVersion': '1.2.3',
 'CFBundleShortVersionString': '1.2.3',
 'CFBundleGetInfoString': 'the demo app',
 'LSApplicationCategoryType': 'public.app-category.developer-tools',
 'LSMinimumSystemVersion': '11.0',
 'NSHighResolutionCapable': True,
 'NSRequiresAquaSystemAppearance': False,
 'NSAppTransportSecurity': {'NSAllowsLocalNetworking': True},
 'LSEnvironment': {'LANG': 'en_US.UTF-8', 'PYTHONUTF8': '1'}}

The macOS plist permits local HTTP, follows the system appearance, and enables UTF-8 before Python starts.

In [ ]:
app.py2app_options()['excludes']

['PyQt5',
 'PyQt6',
 'PySide2',
 'PySide6',
 'clr',
 'gi',
 'matplotlib',
 'py2app',
 'py2exe',
 'setuptools',
 'test',
 'tkinter',
 'webview.platforms.gtk',
 'webview.platforms.winforms',
 'wheel']

The two backends this build is not using are in that list, and the one it is using is not.

In [ ]:
o = app.py2app_options()
o['packages'].append('scipy')
test_eq(app.packages, ['demo', 'fasthtml', 'uvicorn'])
noisy = App(name='D', entry='e.py', excludes=['mlx', 'mlx', 'tkinter']).excluded('darwin')
test_eq(noisy.count('tkinter'), 1)
test_eq(noisy, sorted(set(noisy)))

In [ ]:
#| export
def doc_types(extensions, view_elsewhere=('.svg', '.html', '.htm', '.pdf')):
    "Return Finder document types for folders and owned files."
    exts = sorted({str(e).lstrip('.') for e in extensions if e not in view_elsewhere})
    return [
        {'CFBundleTypeName': 'Folder', 'CFBundleTypeRole': 'Viewer',
         'LSItemContentTypes': ['public.folder'], 'LSHandlerRank': 'Alternate'},
        {'CFBundleTypeName': 'Source file', 'CFBundleTypeRole': 'Editor',
         'CFBundleTypeExtensions': exts, 'LSHandlerRank': 'Owner'},
    ]

`doc_types` keeps folders as alternate handlers and claims non-browser file extensions.

In [ ]:
folder, files = doc_types(['.py', 'py', '.rs', '.svg', '.pdf'])
files['CFBundleTypeExtensions'], folder['LSHandlerRank'], files['LSHandlerRank']

(['py', 'rs'], 'Alternate', 'Owner')

In [ ]:
test_eq(files['CFBundleTypeExtensions'], ['py', 'rs'])
test_eq(folder['LSItemContentTypes'], ['public.folder'])

In [ ]:
#| hide
tmp.cleanup()